# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides an interactive guide for loading and exploring the [FAIR²](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library.

### Dataset Source
The dataset is described and published as a [Croissant](https://mlcommons.org/croissant/) schema at:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load the dataset metadata and instantiate the `mlcroissant.Dataset` object.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Croissant schema URL for the dataset
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load dataset
dataset = mlc.Dataset(croissant_url)

# Access and print basic metadata (using attributes, not dict keys)
print(f"{dataset.metadata.name}: {dataset.metadata.description}")
print(f"Published: {dataset.metadata.datePublished}")
print(f"Identifier: {dataset.metadata.identifier}")
print(f"License: {dataset.metadata.license}")

## 2. Data Overview
Review **available record sets** and their identifiers, fields, and field `@id`s.

The Croissant schema organizes data as one or more 'record sets', each of which may correspond to a table, CSV, or file-like object. Each record set contains 'fields' which map directly to columns or attributes.

In [ ]:
# Inspect all record sets and their fields (@ids)
print("Record sets found in dataset:")
record_sets = list(dataset.metadata.recordSets)
for rs in record_sets:
    print(f"  RecordSet @id: {rs.id}\n    name: {rs.name}")
    print("    Fields:")
    for field in rs.fields:
        print(f"      Field @id: {field.id}")
        print(f"        name: {field.name}")
        print(f"        dataType: {field.dataType}")
    print('-'*40)

Since the names of the record sets and fields are dataset-specific, let's list records from the first record set by its `@id` (replace with another if needed).

In [ ]:
# Display a few records (first record set)
first_record_set = record_sets[0].id
print(f"First record set @id: {first_record_set}")
for i, record in enumerate(dataset.records(record_set=first_record_set)):
    print(record)
    if i >= 2:
        break

## 3. Data Extraction
Extract data from each record set as pandas DataFrames. We use record set and field `@id`s identified above.

In [ ]:
# Extract all record sets into DataFrames, using their @id
dfs = dict()
for rs in record_sets:
    records = list(dataset.records(record_set=rs.id))
    if records:
        df = pd.DataFrame(records)
        dfs[rs.id] = df
        print(f"Loaded DataFrame for RecordSet {rs.id} (shape: {df.shape})")

# Show the columns for the main record set
main_record_set_id = list(dfs.keys())[0]
print(f"Available columns in main record set (@id={main_record_set_id}):")
print(dfs[main_record_set_id].columns.tolist())
dfs[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Let's select a numeric field from the record set for filtering and normalization. **Refer to the Data Overview above to select `@id` of a numeric field** (for example, something like `http://senscience.ai/field/peptide_count`).

For the demo below, we assign the `@id` of one numeric field as `numeric_field_id` and an optional group field as `group_field_id` (based on the published schema and your Data Overview):

In [ ]:
# Example: Use the main record set
record_set_id = main_record_set_id
df = dfs[record_set_id]

# --- Choose field @ids for analysis ---
# Example (update as appropriate for real IDs, see previous Data Overview):
# numeric_field_id = '@id_of_a_numeric_field'  # e.g. 'coverage' or 'peptide_count'
# group_field_id = '@id_of_a_group_field'  # e.g. 'sample_condition' or 'experiment_group'

# For demonstration, do our best to select by 'peptide_count' and 'condition' if present
possible_numeric_fields = [col for col in df.columns if 'count' in col or 'abundance' in col or 'coverage' in col or 'MW' in col or 'Peptide' in col]
print(f"Numeric candidates: {possible_numeric_fields}")
numeric_field = possible_numeric_fields[0] if possible_numeric_fields else df.columns[0]
print(f"Using numeric_field: {numeric_field}")
group_field = None
for candidate in ['sample', 'condition', 'group', 'Protein', 'Description']:
    for col in df.columns:
        if candidate.lower() in col.lower():
            group_field = col
            break
    if group_field is not None:
        break
if group_field:
    print(f"Using group_field: {group_field}")
else:
    print("No obvious group field found; group analysis will be skipped.")

# --- Filter records ---
if pd.api.types.is_numeric_dtype(df[numeric_field]):
    threshold = df[numeric_field].quantile(0.75)
    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f"Filtered records with {numeric_field} > {threshold:.2f}: {len(filtered_df)} records")

    # Normalize
    norm_col = f"{numeric_field}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, norm_col]].head())

    # Group by
    if group_field and group_field in filtered_df.columns and pd.api.types.is_string_dtype(filtered_df[group_field]):
        group_stats = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"Grouped mean {numeric_field} by {group_field}:")
        display(group_stats.head())
else:
    print(f"Selected numeric_field ({numeric_field}) is not numeric. Please review your field selection.")

## 5. Visualization
We'll use matplotlib to visualize the distribution of the selected numeric field and (optionally) the group means. All field references use DataFrame columns derived directly from field `@id`s.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Distribution/histogram of the numeric field
plt.figure(figsize=(8,4))
sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
plt.title(f"Distribution of {numeric_field}")
plt.xlabel(numeric_field)
plt.ylabel("Count")
plt.show()

# Grouped barplot if applicable
if group_field and group_field in df.columns and pd.api.types.is_string_dtype(df[group_field]):
    plt.figure(figsize=(8,4))
    order = df.groupby(group_field)[numeric_field].mean().sort_values(ascending=False).index
    sns.barplot(data=df, x=group_field, y=numeric_field, order=order)
    plt.title(f"Mean {numeric_field} by {group_field}")
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

## 6. Conclusion
This notebook demonstrated how to:

- Load and explore a FAIR dataset using the `mlcroissant` library
- Inspect and reference record sets and fields by their canonical `@id`
- Extract and process data as pandas DataFrames
- Apply filters, normalizations, and group analyses
- Visualize data distributions and group-level summaries

For further analysis, consider adjusting field selections based on your specific research questions and the schema overview provided above.